# Offline Prep (GPT-2 Judge Model)

**Scope**: download GPT-2 medium weights into `/kaggle/working/gpt2-medium/`,
save the notebook output as a Kaggle Dataset (e.g. `gpt2-judge-medium`),
then attach that dataset to consumer notebooks (`prep_explain_for_ppl`,
`prep_ppl_scoring`) which run with **internet OFF**.

**Why**: LN paper Appendix A.2 scores generated text PPL under GPT-2 [Radford et al. 2019].
We mirror that judge model. Medium (355M) matches typical convention; flip
`MODEL_ID` to `gpt2` / `gpt2-large` / `gpt2-xl` for sensitivity if you want.

**Settings**: internet **ON**, CPU is fine (no GPU needed for download). ~1-2 minutes.

**No HF token needed** — GPT-2 is a public model.

## 1. Configuration

Edit `MODEL_ID` if you want a different GPT-2 size. The output directory
name matches HF's identifier, so the consumer notebooks'
`_resolve_judge_path` helper can rglob and find it by substring.

In [ ]:
MODEL_ID  = "gpt2-medium"          # "gpt2" / "gpt2-medium" / "gpt2-large" / "gpt2-xl"
OUTPUT_DIR = f"/kaggle/working/{MODEL_ID}"

print(f"MODEL_ID:   {MODEL_ID}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 2. Download

Uses `huggingface_hub.snapshot_download` (same primitive as `prep/prep_model.ipynb`
uses for Llama). Filters out TF / Flax / JAX checkpoints since we only
need PyTorch weights — GPT-2 ships `pytorch_model.bin` (no safetensors
by default), don't filter `*.bin`.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

from huggingface_hub import snapshot_download
from pathlib import Path

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(out),
    local_dir_use_symlinks=False,
    ignore_patterns=["*.msgpack", "*.h5", "tf_model.h5", "flax_model.msgpack"],
)

for p in sorted(out.iterdir()):
    if p.is_file():
        print(f"  {p.name:40s} {p.stat().st_size / 1e6:>8.1f} MB")

## 3. Verify it loads

Quick sanity check: tokenizer + model load, plus a one-shot PPL on a fluent
English sentence. Expected PPL: ~10-30 for `"The capital of France is Paris."`
under `gpt2-medium`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(out))
mdl = AutoModelForCausalLM.from_pretrained(str(out), torch_dtype=torch.float32)
n_params = sum(p.numel() for p in mdl.parameters())
print(f"loaded {n_params / 1e6:.0f}M params, vocab_size={tok.vocab_size}")

ids = tok("The capital of France is Paris.", return_tensors="pt").input_ids
with torch.no_grad():
    loss = mdl(ids, labels=ids).loss
print(f"smoke PPL on a fluent sentence: {torch.exp(loss).item():.2f}  (expected ~10-30 for medium)")

## 4. Done

**Next**:

1. **Save Version** → Quick Save → "Always save output"
2. After save completes, click **"Output"** tab → **"New Dataset"** button
3. Name it e.g. `gpt2-judge-medium` (private to your account is fine)
4. Attach that dataset as input to `prep_explain_for_ppl.ipynb` and
   `prep_ppl_scoring.ipynb` — they'll auto-discover the model via
   `rglob` in their `_resolve_judge_path` helper.

**To use a different judge size**: rerun this notebook with
`MODEL_ID = "gpt2-large"` (etc.) and save as a separate Kaggle Dataset.
Consumer notebooks pick by substring of `JUDGE_MODEL`.